In [4]:
from typing import Any, List
from dotenv import load_dotenv
import json
from lib.messages import UserMessage, SystemMessage, ToolMessage  # Different message types
from lib.tooling import tool  # Tool decorator for creating AI tools
from lib.llm import LLM  # Our Language Model wrapper

## Building Agent Class


In [17]:
class Agent:
    def __init__(
        self,
        role: str = "You are a helpful assistant.",
        instructions: str = "You are a helpful assistant.",
        model: str = "gpt-4o-mini",
        temperature: float = 0.0,
        tools: List[tool] = None,
    ):
        self.role = role
        self.instructions = instructions
        self.model = model
        self.temperature = temperature
        self.tools = tools if tools is not None else []

        load_dotenv()

        self.llm = LLM(
            model=self.model,
            temperature=self.temperature,
            tools=self.tools,
        )

    def invoke(self, user_message: str) -> str:
        messages = [
            SystemMessage(
                content=(
                    f"You are an AI agent with the role: {self.role}. "
                    f"Your instructions are: {self.instructions}."
                )
            )
        ]

        messages.append(UserMessage(content=user_message))

        ai_message = self.llm.invoke(messages)
        messages.append(ai_message)

        while ai_message.tool_calls:
            for call in ai_message.tool_calls:
                function_name = call.function.name
                function_args = json.loads(call.function.arguments)
                tool_call_id = call.id

                tool = next((t for t in self.tools if t.name == function_name), None)
                if tool:
                    result = tool(**function_args)
                    messages.append(
                        ToolMessage(
                            content=json.dumps(result),
                            tool_call_id=tool_call_id,
                            name=function_name,
                        )
                    )

            ai_message = self.llm.invoke(messages)
            messages.append(ai_message)

        for m in messages:
            print(m)

        return ai_message.content

## Testing Your Agent

Once you've implemented the Agent class, test it with different scenarios:

In [19]:
agent = Agent(role="Coding Assistant")
response = agent.invoke("What is Python? Be concise")
print(response)

content='You are an AI agent with the role: Coding Assistant. Your instructions are: You are a helpful assistant..' role='system'
content='What is Python? Be concise' role='user'
content='Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.' role='assistant' tool_calls=None
Python is a high-level, interpreted programming language known for its readability and simplicity. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python is widely used for web development, data analysis, artificial intelligence, scientific computing, and automation, among other applications.


In [20]:
@tool
def calculate(expression: str) -> float:
    #Evaluate a mathematical expression"""
    return eval(expression)

In [21]:
# Create an agent with the calculator tool
math_agent = Agent(
    role="Math Assistant",
    tools=[calculate]

)

In [22]:
response = math_agent.invoke("What is 23 * 45?")
print(response)

content='You are an AI agent with the role: Math Assistant. Your instructions are: You are a helpful assistant..' role='system'
content='What is 23 * 45?' role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BUqpIm5vYV7LCTbvcGKvq6Rf', function=Function(arguments='{"expression":"23 * 45"}', name='calculate'), type='function')]
content='1035' role='tool' tool_call_id='call_BUqpIm5vYV7LCTbvcGKvq6Rf' name='calculate'
content='The result of \\( 23 \\times 45 \\) is 1035.' role='assistant' tool_calls=None
The result of \( 23 \times 45 \) is 1035.


In [23]:
# Test multiple tool usage
response = math_agent.invoke("If I multiply 3 by 5, what do I get? Then later add 7")
print(response)

content='You are an AI agent with the role: Math Assistant. Your instructions are: You are a helpful assistant..' role='system'
content='If I multiply 3 by 5, what do I get? Then later add 7' role='user'
content=None role='assistant' tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_kG57Pqpq0jQOKeylBASRvATn', function=Function(arguments='{"expression": "3 * 5"}', name='calculate'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_ie0i4Zefd1TQU8NBPu7v5GVQ', function=Function(arguments='{"expression": "(3 * 5) + 7"}', name='calculate'), type='function')]
content='15' role='tool' tool_call_id='call_kG57Pqpq0jQOKeylBASRvATn' name='calculate'
content='22' role='tool' tool_call_id='call_ie0i4Zefd1TQU8NBPu7v5GVQ' name='calculate'
content='If you multiply 3 by 5, you get 15. If you then add 7 to that result, you get 22.' role='assistant' tool_calls=None
If you multiply 3 by 5, you get 15. If you then add 7 to that result, you get 22.
